# Skyline
**Total skyline** represents the cumulative visibility limits derived from regularly spaced observation points within the study area.
## Input
- Digital Elevation Model (DEM)
- Observation points (observationpoints)
## Method
Observation points were generated using the *Fishnet* tool at a regular spacing of 500 m.
For each observation point:
- The skyline was calculated over 360°.
- An azimuth increment of 0.2° was used.
- The maximum visibility distance was set to 8000 m.
- A vertical observer offset of 1.6 m was applied.
The resulting skyline polyline was buffered by 25 m and converted to raster format (25 m cell size).
Finally, all skyline rasters were aggregated using cell-wise summation. The resulting raster therefore represents, for each cell, the number of observation points from which that location forms part of the skyline.

## Implementation instructions
- Add all data under *data/01_Covariate/01.2_Skyline/Skyline.gdb* to the current map
- Adjust input manually (number of iterations, Skyline parameters)
- Execute the cell

In [1]:
# Skyline
import os
import arcpy

# ----------------------- CONFIGURATION -----------------------
# set parallel cores
arcpy.env.processorType = "CPU"
arcpy.env.addOutputsToMap = False
arcpy.env.parallelProcessingFactor = "100%"

# define memory workspace
in_memory_workspace = "memory"
arcpy.env.workspace = in_memory_workspace

# define digital elevation model
in_raster = r"DEM"

# configure environment settings
arcpy.env.extent = in_raster
arcpy.env.cellSize = in_raster
arcpy.env.overwriteOutput = True
    
# create describe object
desc = arcpy.Describe(arcpy.Describe(in_raster).path)
print(desc.path)

# create new file geodatabase
arcpy.management.CreateFileGDB(desc.path,"TotalSkyline.gdb")
out_path = os.path.join(desc.path,"TotalSkyline.gdb")
print(out_path)

# ----------------------- MODELLING -----------------------
## Loop creation
fc = r"observationpoints" #<-- specify the observation points
fields = ['OID@', 'SHAPE@']
n = arcpy.management.GetCount(fc)
with arcpy.da.SearchCursor(fc, fields) as cursor:
   for row in cursor:
        ## create variable of geometry
        point = row[1]
        
        ## create variable for name
        name = "skyline"+str(row[0]).zfill(len(str(n)))
        out_line = os.path.join(out_path,name)
       
        ## create skyline
        arcpy.ddd.Skyline(point,
                          "skyline",
                          in_raster,
                          virtual_surface_radius = "",
                          virtual_surface_elevation = "0 Meters",
                          in_features = None,
                          feature_lod = "FULL_DETAIL",
                          from_azimuth_value_or_field = 0,
                          to_azimuth_value_or_field = 360,
                          azimuth_increment_value_or_field = 0.2, #<-- define azimuth increment
                          max_horizon_radius = "8000 Meters", #<-- define maximum visibility distance
                          segment_skyline = "NO_SEGMENT_SKYLINE",
                          scale_to_percent = 100,
                          scale_according_to = "VERTICAL_ANGLE",
                          scale_method = "SKYLINE_MAXIMUM",
                          use_curvature = "CURVATURE",
                          use_refraction = "REFRACTION",
                          refraction_factor = 0.13,
                          pyramid_level_resolution = 0,
                          create_silhouettes = "NO_CREATE_SILHOUETTES",
                          apply_max_radius_to_features = "NO_APPLY_MAX_RADIUS_TO_FEATURES",
                          vertical_offset = 1.6) #<-- define observation offset             
        
        ## buffer the skyline
        arcpy.analysis.PairwiseBuffer("skyline","buffer",buffer_distance_or_field = "25 Meters",dissolve_option = "NONE",dissolve_field = None,method = "PLANAR",max_deviation = "0 Meters")
        
        ## create raster from buffer
        arcpy.conversion.PolygonToRaster(
            "buffer",
            "OID",
            out_line,
            "CELL_CENTER",
            "NONE",
            25)
       
        print('skyline {} of {}'.format(str(row[0]), n))

# ----------------------- AGGREGATION -----------------------
print('')
print("Aggregating results...")
# set a new environment
arcpy.env.workspace = out_path

# create a list of all remaining rasters
in_rasters = arcpy.ListRasters("skyline*","All")
print("list created")
fcCount = len(in_rasters)

# aggregate all rasters
skyline_raster = arcpy.sa.CellStatistics(in_rasters,"SUM","DATA")
out_result = os.path.join(out_path,"skyline_raster")
skyline_raster.save(out_result)
fcCount = len(in_rasters)
print(fcCount)
print("completed")

C:\Users\ufg\Documents\ArcGIS\Projects\Hilfsprojekt
C:\Users\ufg\Documents\ArcGIS\Projects\Hilfsprojekt\TotalSkyline.gdb
skyline 29058 of 2
skyline 29059 of 2
﻿
﻿
Agregating results...
list created
2
finished
